# 01. 프롬프트 템플릿 만들기

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith('CH02-Prompt')

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [3]:
from langchain_core.prompts import PromptTemplate

template = '{country}의 수도는 어디인가요?'

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [4]:
prompt = prompt.format(country='대한민국')
prompt

'대한민국의 수도는 어디인가요?'

In [6]:
prompt = PromptTemplate.from_template(template)

In [7]:
chain = prompt | llm

chain.invoke('대한민국').content

'대한민국의 수도는 서울특별시입니다.'

In [9]:
template = '{country1}과 {country2}의 수도는 각각 어디인가요?'

prompt = PromptTemplate(
    template=template,
    input_variables=['country1'],
    partial_variables={
        'country2': '미국'
    }
)

prompt

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [10]:
prompt.format(country1='대한민국')

'대한민국과 미국의 수도는 각각 어디인가요?'

In [11]:
prompt_partial = prompt.partial(country2='캐나다')
prompt_partial

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '캐나다'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [12]:
prompt_partial.format(country1='대한민국')

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [13]:
chain = prompt_partial | llm

chain.invoke('대한민국').content

'대한민국의 수도는 서울이고, 캐나다의 수도는 오타와입니다.'

In [14]:
chain.invoke({'country1': '대한민국', 'country2': '호주'}).content

'대한민국의 수도는 서울이며 호주의 수도는 캔버라입니다.'

# 02. 부분 변수 활용하기

In [15]:
from datetime import datetime

datetime.now().strftime('%B %d')

'March 02'

In [16]:
def get_today():
    return datetime.now().strftime('%B %d')

In [17]:
prompt = PromptTemplate(
    template='오늘의 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요.',
    input_variables=['n'],
    partial_variables={
        'today': get_today
    }
)

In [18]:
prompt.format(n=3)

'오늘의 날짜는 March 02입니다. 오늘이 생일인 유명인 3명을 나열해 주세요. 생년월일을 표기해주세요.'

In [19]:
chain = prompt | llm
print(chain.invoke(3).content)

1. 방시혁 (1972년 3월 2일)
2. 다니엘 크레이그 (1968년 3월 2일)
3. 존 보인 (1960년 3월 2일)


In [20]:
print(chain.invoke({'today': 'Jan 02', 'n':3}).content)

1. Taye Diggs - 1971년 1월 2일
2. Kate Bosworth - 1983년 1월 2일
3. Christy Turlington - 1969년 1월 2일


# 03. YAML 파일로부터 프롬프트 템플릿 로드하기

In [21]:
from langchain_core.prompts import load_prompt
prompt = load_prompt('fruit_color.yaml', encoding='utf-8')
prompt

PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [22]:
prompt.format(fruit='사과')

'사과의 색깔이 뭐야?'

In [25]:
prompt2 = load_prompt('capital.yaml', encoding='utf-8')
print(prompt2.format(country='대한민국'))

대한민국의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품
  #Answer:



In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response

chain = prompt2 | ChatOpenAI(model_name='gpt-4o', temperature=0) | StrOutputParser()

answer = chain.stream({'country': '대한민국'})
stream_response(answer)

#Answer:
1. 면적: 서울특별시의 면적은 약 605.21 제곱킬로미터로, 대한민국의 수도이자 가장 큰 도시 중 하나입니다.  
2. 인구: 서울의 인구는 약 950만 명으로, 대한민국에서 가장 인구가 많은 도시입니다.  
3. 역사적 장소: 서울에는 경복궁, 창덕궁, 덕수궁 등 조선시대의 궁궐들이 있으며, 한양도성, 종묘 등 유네스코 세계문화유산으로 지정된 역사적 장소들이 많습니다.  
4. 특산품: 서울은 전통과 현대가 조화를 이루는 도시로, 한복, 한지 공예품, 전통 음식인 김치와 떡 등이 유명합니다. 또한, 현대적인 쇼핑과 문화가 발달하여 다양한 상품과 서비스를 제공합니다.

# 04. ChatPromptTemplate

In [27]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template('{country}의 수도는 어디인가요?')
chat_prompt

ChatPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?'), additional_kwargs={})])

In [28]:
chat_prompt.format(country='대한민국')

'Human: 대한민국의 수도는 어디인가요?'

In [29]:
chat_template = ChatPromptTemplate.from_messages(
    [
        ('system', '당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name}입니다.'),
        ('human', '반가워요!'),
        ('ai', '안녕하세요! 무엇을 도와드릴까요?'),
        ('human', '{user_input}')
    ]
)

In [30]:
messages = chat_template.format_messages(
    name='테디', user_input='당신의 이름은 무엇입니까?'
)
messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='반가워요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [31]:
llm = ChatOpenAI()
llm.invoke(messages).content

'제 이름은 테디입니다. 어떻게 도와드릴까요?'

In [32]:
chain = chat_template | llm

chain.invoke({'name': 'Teddy', 'user_input': '당신의 이름은 무엇입니까?'}).content

'제 이름은 Teddy입니다. 저에게 궁금한 점이 있으면 언제든지 물어보세요!'

# 05. MessagesPlaceholder

In [33]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.'
        ),
        MessagesPlaceholder(variable_name='conversation'),
        ('human', '지금까지의 대화를 {word_count} 단어로 요약합니다.')
    ]
)
chat_prompt

ChatPromptTemplate(input_variables=['conversation', 'word_count'], input_types={'conversation': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annota

In [34]:
formatted_chat_prompt = chat_prompt.format(
    word_count=5,
    conversation=[
        ('human', '안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.'),
        ('ai', '반가워요! 앞으로 잘 부탁드립니다.')
    ]
)

print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.
Human: 안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.
AI: 반가워요! 앞으로 잘 부탁드립니다.
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [35]:
llm = ChatOpenAI()

chain = chat_prompt | llm | StrOutputParser()

In [36]:
chain.invoke(
    {
        'word_count': 5,
        'conversation': [
            (
                'human',
                '안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.'
            ),
            ('ai',
             '반가워요! 앞으로 잘 부탁드립니다.')
        ],
        
    }
)

'신입 테디, 반가워요! 함께 일하게 돼서 기쁩니다.'

# 06. 퓨샷 프롬프트

In [37]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

examples = [
    {
        'question': '스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?',
        'answer': '''이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        '''
    }
]

In [38]:
example_prompt = PromptTemplate.from_template(
    'Question:\n{question}\nAnswer:\n{answer}'
)

print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        
